# Model A Temporal Validation: Original vs. Enriched Initial-Purchase Features

## Purpose

This notebook tests whether the enriched, prediction-time-safe initial-purchase features improve **temporal generalization** for 90-day repeat-purchase prediction. It compares the original eight predictors with the enriched 17-predictor set across the same expanding-window validation folds and the same three baseline model families.

The latest future period is a fixed, untouched final holdout. It is **not** used for preprocessing, model fitting, model selection, threshold selection, or performance evaluation in this notebook.

### How to read this experiment

This is the controlled **model-selection** experiment: Original 8 versus Enriched 17 features across Logistic Regression, Random Forest, and Gradient Boosting using identical expanding temporal-validation folds. The final future holdout is deliberately not loaded with target labels or evaluated here, so development choices cannot be optimized against that future period.

## Experimental guardrails

- Source table: `customer_initial_purchase_model_enriched`.
- Prediction point: the complete initial-purchase event; all predictors were constructed in SQL from information available then.
- Final holdout boundary: inherited from the corrected baseline split rather than reselected for enriched data.
- Development-only temporal validation: three expanding, non-overlapping validation windows.
- No hyperparameter tuning, class weighting, resampling, SMOTE, threshold optimization, or holdout evaluation.
- Average Precision is the primary ranking metric because repeat purchasers are rare; ROC-AUC and top-ranked lift provide complementary evidence.

In [1]:
import math
import os
from getpass import getuser

import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## Load and verify the enriched source table

The SQL checks below document schema, full-table target balance, date coverage, and missingness before modeling. The development query includes labels only through the fixed boundary. The separate holdout query deliberately selects no target column, so final-holdout labels cannot enter development evaluation.

In [2]:
# Define the controlled Original 8 and Enriched 17 comparison before any model is fit. Both sets use
# only initial-purchase-event fields; SQL already enforces the prediction-time and target rules.
# DATABASE_URL makes the notebooks portable without storing a password or machine-specific user name.
# If it is unset, use the documented local PostgreSQL convention for the active operating-system user.
database_url = os.getenv(
    "DATABASE_URL",
    f"postgresql+psycopg2://{getuser()}@localhost:5432/olist_project",
)
engine = create_engine(database_url)

original_features = [
    "customer_state",
    "first_order_amount",
    "freight_to_order_ratio",
    "products_ordered",
    "unique_products_ordered",
    "number_of_categories",
    "number_of_sellers",
    "payment_installments",
]

new_features = [
    "primary_category",
    "payment_type_group",
    "payment_record_count",
    "first_order_month",
    "first_order_weekday",
    "total_product_weight_g",
    "total_product_volume_cm3",
    "any_seller_same_state",
    "avg_customer_seller_distance_km",
]

enriched_features = original_features + new_features
target_column = "repeat_purchase_90d"
time_column = "first_order_date"
id_column = "customer_unique_id"

assert len(original_features) == 8
assert len(enriched_features) == 17

schema = pd.read_sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name = 'customer_initial_purchase_model_enriched'
    ORDER BY ordinal_position
""", engine)
display(schema)

required_columns = set([id_column, target_column, time_column] + enriched_features)
assert required_columns.issubset(set(schema["column_name"]))

source_summary = pd.read_sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(*) FILTER (WHERE repeat_purchase_90d = 'Yes') AS positives,
        COUNT(*) FILTER (WHERE repeat_purchase_90d = 'No') AS negatives,
        100.0 * COUNT(*) FILTER (WHERE repeat_purchase_90d = 'Yes') / COUNT(*) AS positive_rate_pct,
        MIN(first_order_date) AS minimum_first_order_date,
        MAX(first_order_date) AS maximum_first_order_date
    FROM customer_initial_purchase_model_enriched
""", engine)
display(source_summary)

missingness = pd.read_sql("""
    SELECT *
    FROM (
        SELECT 'customer_state' AS feature, COUNT(*) FILTER (WHERE customer_state IS NULL) AS missing_rows FROM customer_initial_purchase_model_enriched
        UNION ALL SELECT 'first_order_amount', COUNT(*) FILTER (WHERE first_order_amount IS NULL) FROM customer_initial_purchase_model_enriched
        UNION ALL SELECT 'freight_to_order_ratio', COUNT(*) FILTER (WHERE freight_to_order_ratio IS NULL) FROM customer_initial_purchase_model_enriched
        UNION ALL SELECT 'products_ordered', COUNT(*) FILTER (WHERE products_ordered IS NULL) FROM customer_initial_purchase_model_enriched
        UNION ALL SELECT 'unique_products_ordered', COUNT(*) FILTER (WHERE unique_products_ordered IS NULL) FROM customer_initial_purchase_model_enriched
        UNION ALL SELECT 'number_of_categories', COUNT(*) FILTER (WHERE number_of_categories IS NULL) FROM customer_initial_purchase_model_enriched
        UNION ALL SELECT 'number_of_sellers', COUNT(*) FILTER (WHERE number_of_sellers IS NULL) FROM customer_initial_purchase_model_enriched
        UNION ALL SELECT 'payment_installments', COUNT(*) FILTER (WHERE payment_installments IS NULL) FROM customer_initial_purchase_model_enriched
        UNION ALL SELECT 'primary_category', COUNT(*) FILTER (WHERE primary_category IS NULL) FROM customer_initial_purchase_model_enriched
        UNION ALL SELECT 'payment_type_group', COUNT(*) FILTER (WHERE payment_type_group IS NULL) FROM customer_initial_purchase_model_enriched
        UNION ALL SELECT 'payment_record_count', COUNT(*) FILTER (WHERE payment_record_count IS NULL) FROM customer_initial_purchase_model_enriched
        UNION ALL SELECT 'first_order_month', COUNT(*) FILTER (WHERE first_order_month IS NULL) FROM customer_initial_purchase_model_enriched
        UNION ALL SELECT 'first_order_weekday', COUNT(*) FILTER (WHERE first_order_weekday IS NULL) FROM customer_initial_purchase_model_enriched
        UNION ALL SELECT 'total_product_weight_g', COUNT(*) FILTER (WHERE total_product_weight_g IS NULL) FROM customer_initial_purchase_model_enriched
        UNION ALL SELECT 'total_product_volume_cm3', COUNT(*) FILTER (WHERE total_product_volume_cm3 IS NULL) FROM customer_initial_purchase_model_enriched
        UNION ALL SELECT 'any_seller_same_state', COUNT(*) FILTER (WHERE any_seller_same_state IS NULL) FROM customer_initial_purchase_model_enriched
        UNION ALL SELECT 'avg_customer_seller_distance_km', COUNT(*) FILTER (WHERE avg_customer_seller_distance_km IS NULL) FROM customer_initial_purchase_model_enriched
    ) AS feature_missingness
    ORDER BY feature
""", engine)
missingness["missing_rate_pct"] = 100 * missingness["missing_rows"] / int(source_summary.loc[0, "rows"])
display(missingness)

,column_name,data_type
0,customer_unique_id,text
1,repeat_purchase_90d,text
2,customer_state,character
3,first_order_date,timestamp without time zone
4,first_order_amount,numeric
5,product_amount,numeric
6,freight_amount,numeric
7,products_ordered,bigint
8,unique_products_ordered,bigint
9,number_of_categories,bigint


,rows,positives,negatives,positive_rate_pct,minimum_first_order_date,maximum_first_order_date
0,86924,1707,85217,1.9638,2016-09-04 21:15:19,2018-07-19 17:24:35


,feature,missing_rows,missing_rate_pct
0,any_seller_same_state,655,0.7535
1,avg_customer_seller_distance_km,1093,1.2574
2,customer_state,0,0.0000
3,first_order_amount,655,0.7535
4,first_order_month,0,0.0000
5,first_order_weekday,0,0.0000
6,freight_to_order_ratio,655,0.7535
7,number_of_categories,655,0.7535
8,number_of_sellers,655,0.7535
9,payment_installments,1,0.0012


## Fixed final holdout

The corrected baseline notebooks used `2018-04-24 09:10:20` as the chronological split timestamp after complete-case filtering. This notebook fixes that same timestamp before looking at enriched-model performance:

- **Development:** `first_order_date <= 2018-04-24 09:10:20`
- **Final holdout:** `first_order_date > 2018-04-24 09:10:20`

Using a fixed timestamp, rather than enriched-data row positions, preserves the same future-period concept while allowing training-only imputation instead of discarding rows with enriched-feature missingness. The holdout query below intentionally omits `repeat_purchase_90d`.

In [3]:
# Fix the future holdout boundary before development modeling. Only pre-boundary rows and labels are
# loaded here; holdout metadata is inspected without target labels so it cannot influence selection.
final_holdout_boundary = pd.Timestamp("2018-04-24 09:10:20")
selected_columns = [id_column, target_column, time_column] + enriched_features

development = pd.read_sql(
    f"""
    SELECT {', '.join(selected_columns)}
    FROM customer_initial_purchase_model_enriched
    WHERE first_order_date <= TIMESTAMP '2018-04-24 09:10:20'
    ORDER BY first_order_date, customer_unique_id
    """,
    engine,
    parse_dates=[time_column],
)

# This query proves holdout population/date coverage without loading its target labels.
final_holdout_metadata = pd.read_sql("""
    SELECT
        COUNT(*) AS rows,
        MIN(first_order_date) AS minimum_first_order_date,
        MAX(first_order_date) AS maximum_first_order_date
    FROM customer_initial_purchase_model_enriched
    WHERE first_order_date > TIMESTAMP '2018-04-24 09:10:20'
""", engine, parse_dates=["minimum_first_order_date", "maximum_first_order_date"])

display(final_holdout_metadata)
assert target_column not in final_holdout_metadata.columns
assert development[time_column].max() <= final_holdout_boundary
assert development[time_column].is_monotonic_increasing
print(f"Development rows available for temporal validation: {len(development):,}")

,rows,minimum_first_order_date,maximum_first_order_date
0,17287,2018-04-24 09:16:29,2018-07-19 17:24:35


Development rows available for temporal validation: 69,637


## Temporal folds

The development period is divided into four chronological blocks at approximately 40%, 60%, and 80% of its ordered rows. The first block initializes training; each following block is used once as validation. Timestamp boundaries are respected so no equal timestamp is split between training and validation. This creates three expanding training windows and three non-overlapping validation windows.

In [4]:
# Expanding folds train on earlier history and validate on the next non-overlapping period. Timestamp
# boundaries keep simultaneous events together, and every model/feature comparison uses these exact folds.
# Compute boundaries from development timestamps only. Customer ID provides deterministic ordering solely for selecting a timestamp.
development = development.sort_values([time_column, id_column]).reset_index(drop=True)
boundaries = [development[time_column].iloc[int(len(development) * share)] for share in (0.40, 0.60, 0.80)]

folds = []
for fold_number, (train_end, validation_end) in enumerate(zip(boundaries, boundaries[1:] + [None]), start=1):
    train_mask = development[time_column] <= train_end
    if validation_end is None:
        validation_mask = development[time_column] > train_end
    else:
        validation_mask = (development[time_column] > train_end) & (development[time_column] <= validation_end)

    train_fold = development.loc[train_mask].copy()
    validation_fold = development.loc[validation_mask].copy()
    assert train_fold[time_column].max() < validation_fold[time_column].min()
    assert set(train_fold[id_column]).isdisjoint(set(validation_fold[id_column]))
    folds.append({
        "fold": fold_number,
        "train": train_fold,
        "validation": validation_fold,
        "train_end": train_end,
        "validation_end": validation_fold[time_column].max(),
    })

def describe_block(name, frame):
    positives = int((frame[target_column] == "Yes").sum())
    return {
        "block": name,
        "rows": len(frame),
        "positives": positives,
        "negatives": len(frame) - positives,
        "positive_rate_pct": 100 * positives / len(frame),
        "minimum_date": frame[time_column].min(),
        "maximum_date": frame[time_column].max(),
    }

fold_description_rows = []
for fold in folds:
    fold_description_rows.append(describe_block(f"Fold {fold['fold']} train", fold["train"]))
    fold_description_rows.append(describe_block(f"Fold {fold['fold']} validation", fold["validation"]))
fold_descriptions = pd.DataFrame(fold_description_rows)
display(fold_descriptions)

assert sum(len(fold["validation"]) for fold in folds) + len(folds[0]["train"]) == len(development)
assert len(set().union(*[set(fold["validation"][id_column]) for fold in folds])) == sum(len(fold["validation"]) for fold in folds)

,block,rows,positives,negatives,positive_rate_pct,minimum_date,maximum_date
0,Fold 1 train,27855,653,27202,2.3443,2016-09-04 21:15:19,2017-10-08 23:34:17
1,Fold 1 validation,13928,275,13653,1.9744,2017-10-08 23:50:17,2017-12-14 23:24:00
2,Fold 2 train,41783,928,40855,2.2210,2016-09-04 21:15:19,2017-12-14 23:24:00
3,Fold 2 validation,13927,287,13640,2.0607,2017-12-14 23:39:28,2018-02-21 22:03:51
4,Fold 3 train,55710,1215,54495,2.1809,2016-09-04 21:15:19,2018-02-21 22:03:51
5,Fold 3 validation,13927,265,13662,1.9028,2018-02-21 22:06:26,2018-04-24 09:10:20


## Feature sets and leakage-safe preprocessing

Both feature sets use the same rows and folds. The original set uses the eight corrected baseline predictors; the enriched set adds the nine initial-purchase features built in SQL.

Numeric columns use a median imputer with missing-value indicators, then standardization. The median and indicators are fitted only on each fold’s training block. Categorical missing values are represented by a literal `missing` token, and `OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=25)` is fitted only on the training block. Thus, rare-category grouping—including `primary_category`—cannot learn validation or holdout frequencies. The 25-row threshold is a fixed compactness rule, not tuned against outcomes.

Product measurements and distance are right-skewed, but this first controlled comparison intentionally applies no `log1p` transform. Keeping transformations simple isolates the incremental value of the enriched fields; a future experiment can assess pre-specified transformations using development folds only.

In [5]:
# Numeric fields receive median imputation plus a missingness indicator; both are learned separately
# inside each training fold. Categorical missing values become an explicit token, while OneHotEncoder groups
# infrequent categories from training data only and safely handles later unseen categories.
categorical_features = [
    "customer_state",
    "primary_category",
    "payment_type_group",
    "first_order_month",
    "first_order_weekday",
    "any_seller_same_state",
]

assert set(categorical_features).issubset(set(enriched_features))

def prepare_feature_frame(frame, feature_set):
    """Apply only deterministic type/missing-token handling before fold-fitted preprocessing."""
    X = frame[feature_set].copy()
    for column in set(feature_set).intersection(categorical_features):
        X[column] = X[column].astype("object").where(X[column].notna(), "missing").astype(str)
    return X

def make_preprocessor(feature_set):
    categorical = [column for column in feature_set if column in categorical_features]
    numeric = [column for column in feature_set if column not in categorical]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=25,
        sparse_output=False,
    )
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric),
            ("categorical", categorical_pipeline, categorical),
        ],
        sparse_threshold=0,
    )

feature_sets = {"Original 8": original_features, "Enriched 17": enriched_features}
for name, features in feature_sets.items():
    print(f"{name}: {len(features)} raw predictors")
    print(features)

Original 8: 8 raw predictors
['customer_state', 'first_order_amount', 'freight_to_order_ratio', 'products_ordered', 'unique_products_ordered', 'number_of_categories', 'number_of_sellers', 'payment_installments']
Enriched 17: 17 raw predictors
['customer_state', 'first_order_amount', 'freight_to_order_ratio', 'products_ordered', 'unique_products_ordered', 'number_of_categories', 'number_of_sellers', 'payment_installments', 'primary_category', 'payment_type_group', 'payment_record_count', 'first_order_month', 'first_order_weekday', 'total_product_weight_g', 'total_product_volume_cm3', 'any_seller_same_state', 'avg_customer_seller_distance_km']


## Evaluation functions

Every fold reports train and validation accuracy, precision, recall, F1, ROC-AUC, Average Precision, prevalence, predicted-positive count at the unchanged 0.50 threshold, and a confusion matrix. Top-1% and top-5% metrics use only probability ranking: they answer how concentrated actual repeat purchasers are among the customers ranked highest by the model. They do not select or optimize a classification threshold.

In [6]:
# Ranking metrics examine the highest-probability customers without changing the default 0.50 classifier.
# Lift is top-K repeat prevalence divided by overall prevalence, so it measures concentration of repeaters.
def ranking_metrics(y_true, probabilities, percentage):
    """Compute deterministic top-K precision, recall, and lift from model probability rankings."""
    y_array = np.asarray(y_true)
    probabilities = np.asarray(probabilities)
    k = max(1, math.ceil(len(y_array) * percentage))
    top_indices = np.argsort(-probabilities, kind="mergesort")[:k]
    positives_in_top_k = int(y_array[top_indices].sum())
    prevalence = float(y_array.mean())
    precision_at_k = positives_in_top_k / k
    recall_at_k = positives_in_top_k / int(y_array.sum()) if y_array.sum() else np.nan
    lift_at_k = precision_at_k / prevalence if prevalence else np.nan
    return {
        f"precision_at_top_{int(percentage * 100)}pct": precision_at_k,
        f"recall_at_top_{int(percentage * 100)}pct": recall_at_k,
        f"lift_at_top_{int(percentage * 100)}pct": lift_at_k,
        f"top_{int(percentage * 100)}pct_count": k,
    }

def evaluate_predictions(y_true, probabilities):
    predictions = (np.asarray(probabilities) >= 0.50).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    metrics = {
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "average_precision": average_precision_score(y_true, probabilities),
        "positive_prevalence": float(np.mean(y_true)),
        "predicted_positives_at_050": int(predictions.sum()),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }
    metrics.update(ranking_metrics(y_true, probabilities, 0.01))
    metrics.update(ranking_metrics(y_true, probabilities, 0.05))
    return metrics

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

## Six controlled experiments

Each model-feature-set combination is rebuilt from scratch for every fold. The preprocessing object is inside the pipeline, so no fitted median, missingness indicator, category frequency, encoder category, scaler parameter, or model parameter crosses from a training block into its validation block.

In [7]:
# Create a fresh Pipeline for every model, feature set, and fold. Pipeline.fit refits imputers, scales,
# and category frequencies only on that fold's historical training rows before validation is transformed.
records = []

for model_name, estimator in models.items():
    for feature_set_name, feature_set in feature_sets.items():
        for fold in folds:
            train_fold = fold["train"]
            validation_fold = fold["validation"]
            X_train = prepare_feature_frame(train_fold, feature_set)
            X_validation = prepare_feature_frame(validation_fold, feature_set)
            y_train = (train_fold[target_column] == "Yes").astype(int)
            y_validation = (validation_fold[target_column] == "Yes").astype(int)

            pipeline = Pipeline([
                ("preprocessor", make_preprocessor(feature_set)),
                ("model", estimator),
            ])
            pipeline.fit(X_train, y_train)

            for split_name, X, y in [
                ("train", X_train, y_train),
                ("validation", X_validation, y_validation),
            ]:
                probabilities = pipeline.predict_proba(X)[:, 1]
                metric_row = evaluate_predictions(y, probabilities)
                metric_row.update({
                    "model": model_name,
                    "feature_set": feature_set_name,
                    "fold": fold["fold"],
                    "split": split_name,
                    "train_end": fold["train_end"],
                    "validation_end": fold["validation_end"],
                })
                records.append(metric_row)

results = pd.DataFrame(records)
assert set(results["split"]) == {"train", "validation"}
assert len(results) == len(models) * len(feature_sets) * len(folds) * 2
print(f"Completed {len(models) * len(feature_sets)} model-feature-set experiments across {len(folds)} temporal folds.")

Completed 6 model-feature-set experiments across 3 temporal folds.


## Fold-level validation results

The table below is the complete validation record for comparing feature sets. Average Precision is primary; values should be interpreted against each fold’s validation prevalence. A high train–validation gap indicates that apparent in-sample fit did not carry forward in time.

In [8]:
# Report every validation fold rather than only a mean so temporal instability and default-threshold behavior remain visible.
validation_results = results.loc[results["split"] == "validation"].copy()
train_results = results.loc[results["split"] == "train"].copy()

fold_level_validation = validation_results[
    [
        "model", "feature_set", "fold", "positive_prevalence",
        "accuracy", "precision", "recall", "f1", "roc_auc", "average_precision",
        "predicted_positives_at_050", "tn", "fp", "fn", "tp",
        "precision_at_top_1pct", "recall_at_top_1pct", "lift_at_top_1pct",
        "precision_at_top_5pct", "recall_at_top_5pct", "lift_at_top_5pct",
    ]
].sort_values(["model", "feature_set", "fold"])
display(fold_level_validation)

,model,feature_set,fold,positive_prevalence,accuracy,precision,recall,f1,roc_auc,average_precision,predicted_positives_at_050,tn,fp,fn,tp,precision_at_top_1pct,recall_at_top_1pct,lift_at_top_1pct,precision_at_top_5pct,recall_at_top_5pct,lift_at_top_5pct
31,Gradient Boosting,Enriched 17,1,0.0197,0.6990,0.0277,0.4182,0.0520,0.5591,0.0267,4148,9620,4033,160,115,0.0143,0.0073,0.7235,0.0416,0.1055,2.1073
33,Gradient Boosting,Enriched 17,2,0.0206,0.9783,0.0000,0.0000,0.0000,0.5868,0.0283,15,13625,15,287,0,0.0214,0.0105,1.0398,0.0344,0.0836,1.6709
35,Gradient Boosting,Enriched 17,3,0.0190,0.9800,0.0000,0.0000,0.0000,0.6027,0.0288,14,13648,14,265,0,0.0429,0.0226,2.2523,0.0344,0.0906,1.8096
25,Gradient Boosting,Original 8,1,0.0197,0.9793,0.0000,0.0000,0.0000,0.5591,0.0242,13,13640,13,275,0,0.0000,0.0000,0.0000,0.0258,0.0655,1.3080
27,Gradient Boosting,Original 8,2,0.0206,0.9791,0.0000,0.0000,0.0000,0.5841,0.0288,4,13636,4,287,0,0.0500,0.0244,2.4263,0.0258,0.0627,1.2532
29,Gradient Boosting,Original 8,3,0.0190,0.9803,0.0000,0.0000,0.0000,0.5869,0.0275,10,13652,10,265,0,0.0286,0.0151,1.5016,0.0430,0.1132,2.2620
7,Logistic Regression,Enriched 17,1,0.0197,0.9803,0.0000,0.0000,0.0000,0.5796,0.0279,0,13653,0,275,0,0.0429,0.0218,2.1706,0.0330,0.0836,1.6713
9,Logistic Regression,Enriched 17,2,0.0206,0.9794,0.0000,0.0000,0.0000,0.5571,0.0274,0,13640,0,287,0,0.0500,0.0244,2.4263,0.0373,0.0906,1.8102
11,Logistic Regression,Enriched 17,3,0.0190,0.9810,0.0000,0.0000,0.0000,0.5989,0.0318,0,13662,0,265,0,0.0500,0.0264,2.6277,0.0502,0.1321,2.6390
1,Logistic Regression,Original 8,1,0.0197,0.9803,0.0000,0.0000,0.0000,0.5468,0.0277,0,13653,0,275,0,0.0571,0.0291,2.8941,0.0387,0.0982,1.9619


## Fold-level training diagnostics

Training results are reported with the same metrics and confusion-matrix components as validation. They are diagnostic only: temporal validation performance, not in-sample fit, determines the development comparison.

In [9]:
# Training-fold metrics are diagnostic. Comparing them with validation metrics helps distinguish weak signal from potential overfitting.
fold_level_training = train_results[
    [
        "model", "feature_set", "fold", "positive_prevalence",
        "accuracy", "precision", "recall", "f1", "roc_auc", "average_precision",
        "predicted_positives_at_050", "tn", "fp", "fn", "tp",
        "precision_at_top_1pct", "recall_at_top_1pct", "lift_at_top_1pct",
        "precision_at_top_5pct", "recall_at_top_5pct", "lift_at_top_5pct",
    ]
].sort_values(["model", "feature_set", "fold"])
display(fold_level_training)

,model,feature_set,fold,positive_prevalence,accuracy,precision,recall,f1,roc_auc,average_precision,predicted_positives_at_050,tn,fp,fn,tp,precision_at_top_1pct,recall_at_top_1pct,lift_at_top_1pct,precision_at_top_5pct,recall_at_top_5pct,lift_at_top_5pct
30,Gradient Boosting,Enriched 17,1,0.0234,0.9781,1.0000,0.0674,0.1263,0.7794,0.2752,44,27202,0,609,44,0.5054,0.2159,21.5578,0.1644,0.3507,7.0125
32,Gradient Boosting,Enriched 17,2,0.0222,0.9790,1.0000,0.0539,0.1022,0.7449,0.2207,50,40855,0,878,50,0.4139,0.1864,18.6347,0.1325,0.2985,5.9674
34,Gradient Boosting,Enriched 17,3,0.0218,0.9792,1.0000,0.0444,0.0851,0.7250,0.1782,54,54495,0,1161,54,0.3315,0.1523,15.2018,0.1177,0.2700,5.3982
24,Gradient Boosting,Original 8,1,0.0234,0.9777,1.0000,0.0490,0.0934,0.7194,0.1936,32,27202,0,621,32,0.3978,0.1700,16.9711,0.1206,0.2573,5.1446
26,Gradient Boosting,Original 8,2,0.0222,0.9785,1.0000,0.0302,0.0586,0.6884,0.1462,28,40855,0,900,28,0.2919,0.1315,13.1412,0.1029,0.2317,4.6317
28,Gradient Boosting,Original 8,3,0.0218,0.9788,1.0000,0.0272,0.0529,0.6728,0.1190,33,54495,0,1182,33,0.2330,0.1070,10.6823,0.0897,0.2058,4.1145
6,Logistic Regression,Enriched 17,1,0.0234,0.9766,0.0000,0.0000,0.0000,0.6762,0.0591,0,27202,0,653,0,0.1219,0.0521,5.1983,0.0711,0.1516,3.0316
8,Logistic Regression,Enriched 17,2,0.0222,0.9778,0.0000,0.0000,0.0000,0.6639,0.0487,0,40855,0,928,0,0.1100,0.0496,4.9549,0.0656,0.1476,2.9514
10,Logistic Regression,Enriched 17,3,0.0218,0.9782,0.0000,0.0000,0.0000,0.6516,0.0439,0,54495,0,1215,0,0.0860,0.0395,3.9442,0.0596,0.1366,2.7320
0,Logistic Regression,Original 8,1,0.0234,0.9766,0.0000,0.0000,0.0000,0.6088,0.0401,0,27202,0,653,0,0.0753,0.0322,3.2107,0.0517,0.1103,2.2048


## Aggregate temporal validation results

Means summarize the three validation periods, while fold-level values and standard deviations expose instability. Train–validation gaps are calculated within the corresponding fold before averaging.

In [10]:
# Aggregate fold-level validation rankings and training-validation gaps. Means summarize typical behavior,
# while standard deviations retain evidence of stability or variation across time.
gap_columns = ["model", "feature_set", "fold"]
gaps = train_results[gap_columns + ["roc_auc", "average_precision"]].merge(
    validation_results[gap_columns + ["roc_auc", "average_precision"]],
    on=gap_columns,
    suffixes=("_train", "_validation"),
)
gaps["roc_auc_gap"] = gaps["roc_auc_train"] - gaps["roc_auc_validation"]
gaps["average_precision_gap"] = gaps["average_precision_train"] - gaps["average_precision_validation"]

summary_rows = []
for (model_name, feature_set_name), group in validation_results.groupby(["model", "feature_set"], sort=False):
    matching_gaps = gaps[(gaps["model"] == model_name) & (gaps["feature_set"] == feature_set_name)]
    ordered = group.sort_values("fold")
    summary_rows.append({
        "model": model_name,
        "feature_set": feature_set_name,
        "mean_validation_average_precision": ordered["average_precision"].mean(),
        "validation_ap_by_fold": ordered["average_precision"].round(4).tolist(),
        "validation_ap_std": ordered["average_precision"].std(ddof=1),
        "mean_validation_roc_auc": ordered["roc_auc"].mean(),
        "validation_roc_auc_by_fold": ordered["roc_auc"].round(4).tolist(),
        "validation_roc_auc_std": ordered["roc_auc"].std(ddof=1),
        "mean_top_1pct_lift": ordered["lift_at_top_1pct"].mean(),
        "top_1pct_lift_by_fold": ordered["lift_at_top_1pct"].round(3).tolist(),
        "mean_top_5pct_lift": ordered["lift_at_top_5pct"].mean(),
        "top_5pct_lift_by_fold": ordered["lift_at_top_5pct"].round(3).tolist(),
        "mean_roc_auc_train_validation_gap": matching_gaps["roc_auc_gap"].mean(),
        "mean_ap_train_validation_gap": matching_gaps["average_precision_gap"].mean(),
    })
summary = pd.DataFrame(summary_rows).sort_values(
    ["mean_validation_average_precision", "mean_validation_roc_auc"], ascending=False
).reset_index(drop=True)
display(summary)

,model,feature_set,mean_validation_average_precision,validation_ap_by_fold,validation_ap_std,mean_validation_roc_auc,validation_roc_auc_by_fold,validation_roc_auc_std,mean_top_1pct_lift,top_1pct_lift_by_fold,mean_top_5pct_lift,top_5pct_lift_by_fold,mean_roc_auc_train_validation_gap,mean_ap_train_validation_gap
0,Logistic Regression,Enriched 17,0.0290,"[0.0279, 0.0274, 0.0318]",0.0024,0.5785,"[0.5796, 0.5571, 0.5989]",0.0209,2.4082,"[2.171, 2.426, 2.628]",2.0402,"[1.671, 1.81, 2.639]",0.0854,0.0216
1,Gradient Boosting,Enriched 17,0.0280,"[0.0267, 0.0283, 0.0288]",0.0011,0.5829,"[0.5591, 0.5868, 0.6027]",0.0221,1.3386,"[0.724, 1.04, 2.252]",1.8626,"[2.107, 1.671, 1.81]",0.1669,0.1967
2,Logistic Regression,Original 8,0.0276,"[0.0277, 0.0254, 0.0297]",0.0022,0.5601,"[0.5468, 0.5503, 0.5831]",0.0200,1.8214,"[2.894, 0.693, 1.877]",1.7967,"[1.962, 1.392, 2.036]",0.0363,0.0087
3,Gradient Boosting,Original 8,0.0268,"[0.0242, 0.0288, 0.0275]",0.0024,0.5767,"[0.5591, 0.5841, 0.5869]",0.0153,1.3093,"[0.0, 2.426, 1.502]",1.6077,"[1.308, 1.253, 2.262]",0.1168,0.1261
4,Random Forest,Enriched 17,0.0243,"[0.0194, 0.0276, 0.026]",0.0043,0.5381,"[0.4747, 0.5779, 0.5616]",0.0554,2.0510,"[0.724, 2.426, 3.003]",1.4839,"[1.163, 1.253, 2.036]",0.4619,0.9756
5,Random Forest,Original 8,0.0219,"[0.0215, 0.0237, 0.0205]",0.0017,0.5230,"[0.5179, 0.5449, 0.5063]",0.0198,1.8118,"[2.894, 1.04, 1.502]",1.3784,"[0.945, 1.532, 1.659]",0.4739,0.9126


## Original 8 versus Enriched 17

Feature-set differences are paired by model family and temporal fold. Positive values mean that the enriched features improved that validation metric relative to the original eight features on the same customers and time window.

In [11]:
# Pair Original 8 and Enriched 17 within the same model and fold. This isolates feature enrichment from
# changes in time window, estimator, or preprocessing fit.
paired = validation_results.pivot(
    index=["model", "fold"],
    columns="feature_set",
    values=["average_precision", "roc_auc", "lift_at_top_1pct", "lift_at_top_5pct"],
)
paired.columns = [f"{metric}_{feature_set}" for metric, feature_set in paired.columns]
paired = paired.reset_index()
for metric in ["average_precision", "roc_auc", "lift_at_top_1pct", "lift_at_top_5pct"]:
    paired[f"{metric}_enriched_minus_original"] = (
        paired[f"{metric}_Enriched 17"] - paired[f"{metric}_Original 8"]
    )
display(paired.sort_values(["model", "fold"]))

comparison = paired.groupby("model", as_index=False)[
    [
        "average_precision_enriched_minus_original",
        "roc_auc_enriched_minus_original",
        "lift_at_top_1pct_enriched_minus_original",
        "lift_at_top_5pct_enriched_minus_original",
    ]
].agg(["mean", "std"])
comparison.columns = ["model"] + [f"{metric}_{stat}" for metric, stat in comparison.columns[1:]]
display(comparison)

,model,fold,average_precision_Enriched 17,average_precision_Original 8,roc_auc_Enriched 17,roc_auc_Original 8,lift_at_top_1pct_Enriched 17,lift_at_top_1pct_Original 8,lift_at_top_5pct_Enriched 17,lift_at_top_5pct_Original 8,average_precision_enriched_minus_original,roc_auc_enriched_minus_original,lift_at_top_1pct_enriched_minus_original,lift_at_top_5pct_enriched_minus_original
0,Gradient Boosting,1,0.0267,0.0242,0.5591,0.5591,0.7235,0.0000,2.1073,1.3080,0.0025,0.0000,0.7235,0.7993
1,Gradient Boosting,2,0.0283,0.0288,0.5868,0.5841,1.0398,2.4263,1.6709,1.2532,-0.0004,0.0027,-1.3865,0.4177
2,Gradient Boosting,3,0.0288,0.0275,0.6027,0.5869,2.2523,1.5016,1.8096,2.2620,0.0013,0.0158,0.7508,-0.4524
3,Logistic Regression,1,0.0279,0.0277,0.5796,0.5468,2.1706,2.8941,1.6713,1.9619,0.0002,0.0328,-0.7235,-0.2907
4,Logistic Regression,2,0.0274,0.0254,0.5571,0.5503,2.4263,0.6932,1.8102,1.3924,0.0020,0.0068,1.7331,0.4177
5,Logistic Regression,3,0.0318,0.0297,0.5989,0.5831,2.6277,1.8770,2.6390,2.0358,0.0021,0.0158,0.7508,0.6032
6,Random Forest,1,0.0194,0.0215,0.4747,0.5179,0.7235,2.8941,1.1626,0.9446,-0.0021,-0.0432,-2.1706,0.2180
7,Random Forest,2,0.0276,0.0237,0.5779,0.5449,2.4263,1.0398,1.2532,1.5317,0.0038,0.0330,1.3865,-0.2785
8,Random Forest,3,0.0260,0.0205,0.5616,0.5063,3.0031,1.5016,2.0358,1.6588,0.0055,0.0552,1.5016,0.3770


,model,average_precision_enriched_minus_original_mean,average_precision_enriched_minus_original_std,roc_auc_enriched_minus_original_mean,roc_auc_enriched_minus_original_std,lift_at_top_1pct_enriched_minus_original_mean,lift_at_top_1pct_enriched_minus_original_std,lift_at_top_5pct_enriched_minus_original_mean,lift_at_top_5pct_enriched_minus_original_std
0,Gradient Boosting,0.0011,0.0015,0.0062,0.0084,0.0293,1.2261,0.2549,0.6416
1,Logistic Regression,0.0014,0.0011,0.0184,0.0132,0.5868,1.2365,0.2434,0.4717
2,Random Forest,0.0024,0.0040,0.0150,0.0516,0.2391,2.0877,0.1055,0.3419


## Generalization diagnosis and development recommendation

A flexible model can look strong on its training window yet fail on later customers. The diagnostics below keep train–validation gaps separate from absolute validation ranking performance. The recommendation is based only on development-fold Average Precision, ROC-AUC, top-ranked lift, and temporal stability. It does not use the final holdout.

In [12]:
# Select from development-period temporal-validation evidence only. This summary is descriptive and
# deliberately does not inspect final-future-holdout outcomes.
diagnostics = summary[[
    "model", "feature_set", "mean_validation_average_precision",
    "mean_validation_roc_auc", "mean_top_1pct_lift",
    "mean_roc_auc_train_validation_gap", "mean_ap_train_validation_gap",
    "validation_ap_std", "validation_roc_auc_std",
]].copy()

def generalization_label(row):
    if row["mean_ap_train_validation_gap"] >= 0.10 or row["mean_roc_auc_train_validation_gap"] >= 0.20:
        return "substantial overfitting risk"
    if row["mean_validation_average_precision"] <= 1.25 * validation_results["positive_prevalence"].mean():
        return "weak ranking signal"
    return "mixed / requires stability review"

diagnostics["diagnosis"] = diagnostics.apply(generalization_label, axis=1)
display(diagnostics)

recommended_development_configuration = summary.iloc[0][["model", "feature_set"]].to_dict()
print("Development-only ranking leader (not final-holdout evaluated):", recommended_development_configuration)
print("Final-holdout target labels were not loaded or evaluated in this notebook.")

,model,feature_set,mean_validation_average_precision,mean_validation_roc_auc,mean_top_1pct_lift,mean_roc_auc_train_validation_gap,mean_ap_train_validation_gap,validation_ap_std,validation_roc_auc_std,diagnosis
0,Logistic Regression,Enriched 17,0.0290,0.5785,2.4082,0.0854,0.0216,0.0024,0.0209,mixed / requires stability review
1,Gradient Boosting,Enriched 17,0.0280,0.5829,1.3386,0.1669,0.1967,0.0011,0.0221,substantial overfitting risk
2,Logistic Regression,Original 8,0.0276,0.5601,1.8214,0.0363,0.0087,0.0022,0.0200,mixed / requires stability review
3,Gradient Boosting,Original 8,0.0268,0.5767,1.3093,0.1168,0.1261,0.0024,0.0153,substantial overfitting risk
4,Random Forest,Enriched 17,0.0243,0.5381,2.0510,0.4619,0.9756,0.0043,0.0554,substantial overfitting risk
5,Random Forest,Original 8,0.0219,0.5230,1.8118,0.4739,0.9126,0.0017,0.0198,substantial overfitting risk


Development-only ranking leader (not final-holdout evaluated): {'model': 'Logistic Regression', 'feature_set': 'Enriched 17'}
Final-holdout target labels were not loaded or evaluated in this notebook.


## Conclusion boundary

This notebook establishes a leakage-safe development comparison and identifies a configuration to carry forward only after review. It intentionally does **not** report final-holdout ROC-AUC, Average Precision, classification metrics, predictions, or target-label analysis. The final holdout remains untouched for one later confirmation evaluation.